# Crestbound Duelists — Balance Analysis

**Optimal decision-making under uncertainty in adversarial turn-based systems.**

This notebook runs the full 6×6 Monte Carlo simulation matrix, compares AI policies,
and visualises the results. The game engine is a turn-based class combat system with
six classes, three move archetypes (Basic / Signature / Gambit), probabilistic speed
resolution, and a compressed damage formula `2·ATK/(ATK+DEF)`.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the game package is importable
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

from models import ClassName, CLASS_STATS, create_unit
from simulation import generate_win_matrix, compute_averages, run_matchup
from ai import POLICIES

sns.set_theme(style="darkgrid")
CLASS_NAMES = [c.value for c in ClassName]
N_SIMS = 10_000  # per matchup — increase to 100k for final results
print(f"Running {N_SIMS:,} sims per matchup ({N_SIMS * 15:,} total battles)")

## 1. Greedy AI — Win-Rate Matrix

Each cell shows the win rate (%) of the **row** class against the **column** class,
averaged over N simulations with both sides using the Greedy AI policy.

In [ ]:
# Generate the full 6×6 matrix
matrix_greedy = generate_win_matrix(n_sims=N_SIMS, policy_name="greedy", verbose=False)
avgs_greedy = compute_averages(matrix_greedy)

# Convert to numpy array for heatmap
classes = list(ClassName)
data = np.zeros((6, 6))
for i, a in enumerate(classes):
    for j, b in enumerate(classes):
        data[i, j] = matrix_greedy[(a, b)]

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.eye(6, dtype=bool)
sns.heatmap(
    data, annot=True, fmt=".1f", mask=mask,
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    cmap="RdYlGn", center=50, vmin=0, vmax=100,
    linewidths=0.5, linecolor="#333",
    cbar_kws={"label": "Win Rate (%)"},
    ax=ax
)
ax.set_title("Greedy AI — Win-Rate Matrix", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Defender")
ax.set_ylabel("Attacker")
plt.tight_layout()
plt.savefig("results/greedy_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Class Rankings

Average win rate across all non-mirror matchups. A perfectly balanced game
would have all classes at exactly 50%.

In [ ]:
# Sort by average win rate
ranked = sorted(avgs_greedy.items(), key=lambda x: x[1], reverse=True)
names = [c.value for c, _ in ranked]
rates = [r for _, r in ranked]
deltas = [r - 50 for r in rates]
colors = ["#22c55e" if d > 0 else "#ef4444" if d < 0 else "#eab308" for d in deltas]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(names[::-1], rates[::-1], color=colors[::-1], edgecolor="#333", linewidth=0.5)
ax.axvline(50, color="#666", linestyle="--", linewidth=1, label="Perfect balance (50%)")
ax.set_xlim(30, 70)
ax.set_xlabel("Average Win Rate (%)")
ax.set_title("Class Balance Rankings — Greedy AI", fontsize=14, fontweight="bold")

for bar, rate, delta in zip(bars, rates[::-1], deltas[::-1]):
    sign = "+" if delta > 0 else ""
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{rate:.1f}% ({sign}{delta:.1f})", va="center", fontsize=10)

ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("results/class_rankings.png", dpi=150, bbox_inches="tight")
plt.show()

spread = max(rates) - min(rates)
print(f"Balance spread: {spread:.1f}pp (lower is better)")

## 3. Lookahead AI — Does Strategy Matter?

We run the same matrix with the Lookahead AI (1-step minimax) and compare
the class rankings to see if strategic depth changes the balance landscape.

In [ ]:
matrix_lookahead = generate_win_matrix(n_sims=N_SIMS, policy_name="lookahead", verbose=False)
avgs_lookahead = compute_averages(matrix_lookahead)

# Compare
comparison = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Greedy": [avgs_greedy[c] for c in classes],
    "Lookahead": [avgs_lookahead[c] for c in classes],
})
comparison["Delta"] = comparison["Lookahead"] - comparison["Greedy"]
comparison = comparison.sort_values("Greedy", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(comparison))
w = 0.35
ax.bar(x - w/2, comparison["Greedy"], w, label="Greedy", color="#3b82f6", edgecolor="#333")
ax.bar(x + w/2, comparison["Lookahead"], w, label="Lookahead", color="#a855f7", edgecolor="#333")
ax.axhline(50, color="#666", linestyle="--", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(comparison["Class"])
ax.set_ylabel("Average Win Rate (%)")
ax.set_title("Policy Comparison — Greedy vs Lookahead", fontsize=14, fontweight="bold")
ax.legend()
ax.set_ylim(30, 70)
plt.tight_layout()
plt.savefig("results/policy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(comparison.to_string(index=False))

## 4. Fight Duration Analysis

Average number of turns per matchup — reveals which matchups are quick kills
vs extended wars of attrition.

In [ ]:
turns_data = np.zeros((6, 6))
for i, a in enumerate(classes):
    for j, b in enumerate(classes):
        if i == j:
            turns_data[i, j] = 0
            continue
        if i > j:
            turns_data[i, j] = turns_data[j, i]
            continue
        r = run_matchup(a, b, 2000, "greedy")
        turns_data[i, j] = r["avg_turns"]
        turns_data[j, i] = r["avg_turns"]

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.eye(6, dtype=bool)
sns.heatmap(
    turns_data, annot=True, fmt=".1f", mask=mask,
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    cmap="YlOrRd", linewidths=0.5, linecolor="#333",
    cbar_kws={"label": "Avg Turns"},
    ax=ax
)
ax.set_title("Average Fight Duration (Turns)", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("results/fight_duration.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Move Usage Distribution

Which moves does the Greedy AI actually choose? This reveals whether
the Signature and Gambit moves are strategically viable or if Basic
spam dominates.

In [ ]:
move_data = {}
for cls in classes:
    # Run mirror matches to see move selection patterns
    counts = {}
    for opp in classes:
        if cls == opp:
            continue
        r = run_matchup(cls, opp, 1000, "greedy", log_actions=True)
        for log in r["logs"]:
            if log.actor_class == cls.value:
                slot = log.move_slot
                counts[slot] = counts.get(slot, 0) + 1
    total = sum(counts.values())
    move_data[cls.value] = {k: round(100 * v / total, 1) for k, v in counts.items()} if total > 0 else {}

fig, ax = plt.subplots(figsize=(9, 5))
slots = ["BASIC", "SIGNATURE", "GAMBIT"]
slot_colors = {"BASIC": "#64748b", "SIGNATURE": "#a855f7", "GAMBIT": "#ef4444"}
x = np.arange(len(CLASS_NAMES))
bottom = np.zeros(len(CLASS_NAMES))

for slot in slots:
    vals = [move_data.get(c, {}).get(slot, 0) for c in CLASS_NAMES]
    ax.bar(x, vals, bottom=bottom, label=slot, color=slot_colors[slot], edgecolor="#333", linewidth=0.5)
    bottom += np.array(vals)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel("Usage (%)")
ax.set_title("Move Slot Usage by Class — Greedy AI", fontsize=14, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("results/move_usage.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| **Classes** | 6 (Warrior, Mage, Assassin, Guardian, Neutral, Sorcerer) |
| **Moves per class** | 3 (Basic / Signature / Gambit) |
| **Damage formula** | `floor(Power × 2·ATK/(ATK+DEF) × v)`, v ~ U(0.85, 1.0) |
| **Speed system** | Probabilistic band (B=7) with guaranteed threshold at 2× ratio |
| **AI policies** | Random, Greedy (expected-value), Lookahead (1-step minimax) |
| **Balance spread** | See rankings above — target is <15pp |

### Key Findings

1. **Compressed formula works** — fights last 3–5 turns instead of 2HKO determinism
2. **Greedy vs Lookahead delta is small** — the move space is compact enough that
   expected-value play is near-optimal; strategic depth shows more in buff/debuff timing
3. **Neutral's adaptive advantage** is real but controlled — Focus Shift creates a
   risk/reward window that differentiates it from pure specialists
4. **Mage vs Assassin** remains the hardest counter — structural type mismatch
   (physical-only into high-RES) with no current workaround

### Next Steps

- Nash equilibrium solver for mixed-strategy optimal play
- 3v3 team composition with TFT-style synergy traits
- Extended action logging for reinforcement learning experiments